In [2]:
import pandas as pd

In [55]:
train = 'https://raw.githubusercontent.com/ichiP245/TP_Kaggle_Predictivo/refs/heads/main/base_train.csv'

In [56]:
df = pd.read_csv(train)

In [57]:
pd.set_option('display.max_columns', None)

In [77]:
df['track_popularity'].value_counts().head(3)

,count
track_popularity,
0,1900
60,464
57,463


In [61]:
val = 'https://raw.githubusercontent.com/ichiP245/TP_Kaggle_Predictivo/refs/heads/main/base_val.csv'
df_val = pd.read_csv(val)

Prediccion 1: todos 0

In [62]:
df_val['pred_0'] = 0

In [64]:
df_val[['Unnamed: 0', 'pred_0']].rename(columns={'Unnamed: 0':'ID', 'pred_0':'track_popularity'}).to_csv('first_pred.csv',index=False)

Prediccion 2: RandomForestRegressor con variables numericas

In [65]:
import numpy as np
from sklearn.ensemble import RandomForestRegressor

# Define features and target
numerical_features = [
    'danceability', 'energy', 'key', 'loudness', 'mode',
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'duration_ms'
]
target = 'track_popularity'

# Calculate the overall mean popularity from the training data
overall_mean_popularity = df[target].mean()

pred_2 = df.groupby('playlist_subgenre')['track_popularity'].mean().to_dict()

# Add 'mean_subgenre_popularity' as a feature to df (training data)
df['mean_subgenre_popularity'] = df['playlist_subgenre'].map(pred_2).fillna(overall_mean_popularity)

# Add 'mean_subgenre_popularity' as a feature to df_val (validation data)
# Use .fillna() to handle subgenres in df_val that might not have been present in the training set
df_val['mean_subgenre_popularity'] = df_val['playlist_subgenre'].map(pred_2).fillna(overall_mean_popularity)

# Combine numerical and engineered features
features = numerical_features + ['mean_subgenre_popularity']

# Prepare the data for training
X_train_full = df[features]
y_train_full = df[target]

# Initialize the Random Forest Regressor
# Using n_estimators=100 and random_state=42 for reproducibility, n_jobs=-1 for faster computation
rfr = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)

# Train the model on the full training data
print("Training Random Forest Regressor...")
rfr.fit(X_train_full, y_train_full)
print("Training complete.")

Training Random Forest Regressor...
Training complete.


In [68]:
# Prepare the validation data for prediction
X_val = df_val[features]

# Make predictions on the validation set
print("Making predictions on the validation set...")
predictions = rfr.predict(X_val)

Making predictions on the validation set...


In [73]:
# Add predictions to df_val
df_val['track_popularity_predicted'] = predictions

In [75]:
df_val[['Unnamed: 0', 'track_popularity_predicted']].rename(columns={'Unnamed: 0':'ID', 'track_popularity_predicted':'track_popularity'}).to_csv('second_pred.csv',index=False)

Prediccion 3: CatBoost Regressor

In [82]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.2 MB/s eta 0:00:00


In [84]:
from catboost import CatBoostRegressor
import numpy as np
import pandas as pd # Ensure pandas is imported if not already in this cell

# Define features
numerical_features = [
    'danceability', 'energy', 'key', 'loudness', 'mode',
    'speechiness', 'acousticness', 'instrumentalness',
    'liveness', 'valence', 'tempo', 'duration_ms'
]
engineered_numerical_feature = 'mean_subgenre_popularity'
categorical_features = ['playlist_genre', 'playlist_subgenre']
target = 'track_popularity'

# Combine all features for CatBoost
all_features = numerical_features + [engineered_numerical_feature] + categorical_features

# Prepare the data for training
X_train_cb = df[all_features]
y_train_cb = df[target]

# Prepare the validation data
X_val_cb = df_val[all_features]

# Identify categorical feature names for CatBoost
cb_categorical_features = [col for col in categorical_features if col in X_train_cb.columns]

# Initialize CatBoost Regressor with some optimized parameters
# These values are examples and can be further tuned using methods like GridSearchCV
cbr = CatBoostRegressor(
    random_seed=42,
    verbose=0, # Suppress verbose output
    iterations=2000, # Increased iterations for potentially better performance
    learning_rate=0.05, # Example learning rate
    depth=8, # Example tree depth
    l2_leaf_reg=3 # Example L2 regularization
)

# Train the model
print("Training CatBoost Regressor with optimized parameters...")
cbr.fit(X_train_cb, y_train_cb, cat_features=cb_categorical_features)
print("Training complete.")

# Make predictions on the validation set
print("Making predictions on the validation set...")
predictions_cb = cbr.predict(X_val_cb)

# Clip predictions to the valid range of track_popularity (0-100)
predictions_cb = np.clip(predictions_cb, 0, 100)

# Create the submission DataFrame using 'Unnamed: 0' as ID
submission_df_cb = pd.DataFrame({
    'ID': df_val['Unnamed: 0'],
    'track_popularity': predictions_cb
})

# Save the submission file
submission_df_cb.to_csv('pred_catboost_optimized.csv', index=False) # Changed filename to reflect optimization

print("Submission file 'pred_catboost_optimized.csv' created successfully!")
print(submission_df_cb.head())

Training CatBoost Regressor with optimized parameters...
Training complete.
Making predictions on the validation set...
Submission file 'pred_catboost_optimized.csv' created successfully!
      ID  track_popularity
0  26266         34.870093
1  26267         32.234943
2  26268         31.578905
3  26269         28.543089
4  26270         17.270679
